# Is Spain in a vibecession?

In 2022 the writer Kyla Scanlon coined the word **vibecession** for a puzzle in the United States: the hard data looked fine (low unemployment, growing output), yet consumer sentiment was as gloomy as in a recession.

Spain in 2025 and 2026 looks like a candidate for the same puzzle. By the official numbers the economy has been one of the strongest in Europe: GDP growth well above the euro area, record employment, unemployment at its lowest since 2008, and a minimum wage that has outpaced inflation. Yet surveys and street conversations are dominated by housing costs, rents, and the price of the weekly shop.

In this notebook we ask the question with data:

1. **Source** monthly data from Eurostat: consumer confidence, unemployment, prices by category, interest rates and house prices, for Spain and the euro area.
2. **Build** one tidy monthly dataset and save it to CSV.
3. **Train** machine learning models that predict consumer confidence from economic fundamentals, using only the years 2000 to 2019.
4. **Predict** confidence after 2020 and measure the **vibes gap**: the difference between how people say they feel and how the fundamentals say they *should* feel.
5. **Explain** the gap: does adding housing and food prices close it? Is Spain's gap bigger than the euro area's?

A good answer might be "no, Spain is not in a vibecession". That is a perfectly fine result. The point is to let the data answer.

## 0. Setup

This notebook runs in Google Colab, JupyterLite, or any local Jupyter. In JupyterLite (which runs Python inside your browser) the next cell installs the packages first.

In [ ]:
import sys

if sys.platform == "emscripten":  # running inside the browser (JupyterLite / Pyodide)
    import piplite
    await piplite.install(["polars", "scikit-learn", "matplotlib"])

In [ ]:
#!pip install polars

In [ ]:
import json
import os
import re
import urllib.parse
from datetime import date

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_width_chars(200)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})

ES_COLOR, EA_COLOR, MODEL_COLOR = "#c0392b", "#2c5aa0", "#7f8c8d"

## 1. Sourcing the data

Everything comes from the [Eurostat database](https://ec.europa.eu/eurostat/web/main/data/database). Its API is free, needs no key, and returns data in a format called JSON-stat: a flat list of numbers plus a description of the dimensions (country, indicator, month...) that tells you how to unfold them into a table.

| What | Eurostat dataset | Why it matters |
|---|---|---|
| Consumer confidence and its components | `ei_bsco_m` | The "vibes": a monthly household survey run in every EU country |
| Unemployment rate | `une_rt_m` | The classic driver of sentiment |
| Consumer prices by category | `prc_hicp_midx` | All items, food, rents, energy |
| Euribor (3 and 12 month) | `irt_st_m` | Most Spanish mortgages are variable-rate, tied to 12-month Euribor |
| House prices | `prc_hpi_q` | Quarterly; Spain from 2007 |

The first function fetches a URL in a way that works both in normal Python and inside the browser. The second turns a JSON-stat response into a Polars DataFrame.

In [ ]:
BASE = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"


def fetch_json(url):
    if sys.platform == "emscripten":
        from pyodide.http import open_url  # browser-friendly, synchronous request
        return json.load(open_url(url))
    import urllib.request
    with urllib.request.urlopen(url, timeout=120) as resp:
        return json.load(resp)


def eurostat(dataset, **filters):
    # Fetch a Eurostat dataset and return a tidy DataFrame: one row per observation
    params = {"lang": "EN", "format": "JSON", **filters}
    js = fetch_json(BASE + dataset + "?" + urllib.parse.urlencode(params, doseq=True))

    dims, sizes = js["id"], js["size"]
    codes = []  # for each dimension, position -> category code
    for d in dims:
        idx = js["dimension"][d]["category"]["index"]
        inv = [None] * len(idx)
        for code_, pos in (idx.items() if isinstance(idx, dict) else enumerate(idx)):
            inv[pos] = code_
        codes.append(inv)

    values = js["value"]
    rows = []
    for flat, v in (values.items() if isinstance(values, dict) else enumerate(values)):
        if v is None:
            continue
        flat, rec = int(flat), {}
        for d, size, inv in reversed(list(zip(dims, sizes, codes))):  # unfold, last dimension first
            flat, pos = divmod(flat, size)
            rec[d] = inv[pos]
        rec["value"] = float(v)
        rows.append(rec)
    return pl.DataFrame(rows).select(dims + ["value"])

We request each dataset one country at a time, so that a missing code for one country does not sink the whole request. For the euro area we try the fixed 20-country aggregate first and fall back to Eurostat's evolving-composition aggregate (`EA`).

**If a request fails:** Eurostat occasionally renames datasets or codes. In 2026 it began moving consumer price data to a new classification (COICOP 2018), so the price request is the most likely to need a tweak. Look the dataset up in the Eurostat data browser and edit the code in the `REQUESTS` cell.

In [ ]:
GEOS = ["ES", "EA20", "EA"]

CONF_INDICATORS = {
    "BS-CSMCI": "conf",           # consumer confidence indicator (headline)
    "BS-FS-LY": "fin_past",       # own financial situation, past 12 months
    "BS-FS-NY": "fin_next",       # own financial situation, next 12 months
    "BS-GES-NY": "econ_next",     # general economic situation, next 12 months
    "BS-PT-LY": "prices_felt",    # perceived price trends, past 12 months
    "BS-UE-NY": "unemp_fear",     # unemployment expectations, next 12 months
}
PRICE_ITEMS = {"CP00": "hicp", "CP01": "food", "CP041": "rent", "NRG": "energy"}

REQUESTS = {
    "conf":  ("ei_bsco_m",     dict(indic=list(CONF_INDICATORS), s_adj="SA", unit="BAL")),
    "unemp": ("une_rt_m",      dict(s_adj="SA", age="TOTAL", sex="T", unit="PC_ACT")),
    "hicp":  ("prc_hicp_midx", dict(coicop=list(PRICE_ITEMS))),
    "hpi":   ("prc_hpi_q",     dict(purchase="TOTAL")),
}

raw = {}
for name, (ds, f) in REQUESTS.items():
    parts = []
    for g in GEOS:
        try:
            parts.append(eurostat(ds, geo=g, **f))
        except Exception as e:
            print(f"  {name}/{g}: not available ({type(e).__name__})")
    if parts:
        raw[name] = pl.concat(parts, how="vertical_relaxed")
        print(f"{name:6s} {ds:14s} {raw[name].height:6d} rows, geos: {sorted(raw[name]['geo'].unique())}")

try:
    raw["rates"] = eurostat("irt_st_m", geo="EA", int_rt=["IRT_M3", "IRT_M12"])
    print(f"rates  irt_st_m       {raw['rates'].height:6d} rows")
except Exception as e:
    print("rates FAILED:", e)

EA = next((g for g in ["EA20", "EA"] if "conf" in raw and g in raw["conf"]["geo"].to_list()), "EA20")
print("Euro area code used:", EA)

## 2. Building one monthly dataset

Each series arrives with its own time format (`2024-03` for months, `2024-Q1` for quarters) and sometimes more than one unit or index base. The `tidy` helper picks out one series, converts the period to a date, and, if a dimension still has several categories (say, two index bases), keeps one and tells you which.

In [ ]:
def parse_period(t):
    m = re.match(r"^(\d{4})-?M?(\d{2})$", t)
    if m:
        return date(int(m[1]), int(m[2]), 1)
    m = re.match(r"^(\d{4})-?Q(\d)$", t)
    if m:
        return date(int(m[1]), 3 * int(m[2]) - 2, 1)
    return date(int(t[:4]), 1, 1)


def tidy(df, name, **where):
    for k, v in where.items():
        df = df.filter(pl.col(k) == v)
    for c in df.columns:
        if c not in ("geo", "time", "value") and df[c].n_unique() > 1:
            keep = sorted(df[c].unique().to_list())[0]
            print(f"  note: {name} has several '{c}' categories, keeping {keep}")
            df = df.filter(pl.col(c) == keep)
    return df.select(
        pl.col("time").map_elements(parse_period, return_dtype=pl.Date).alias("date"),
        pl.col("value").alias(name),
    ).sort("date")

In [ ]:
def yoy(col, lag=12):
    # Year-on-year percent change
    return (pl.col(col) / pl.col(col).shift(lag) - 1) * 100


def build(geo):
    months = pl.DataFrame({"date": pl.date_range(date(1995, 1, 1), date.today().replace(day=1), "1mo", eager=True)})
    df = months

    for indic, col in CONF_INDICATORS.items():
        df = df.join(tidy(raw["conf"], col, geo=geo, indic=indic), on="date", how="left")
    df = df.join(tidy(raw["unemp"], "unemp", geo=geo), on="date", how="left")
    for item, col in PRICE_ITEMS.items():
        if "hicp" in raw and raw["hicp"].filter((pl.col("geo") == geo) & (pl.col("coicop") == item)).height:
            df = df.join(tidy(raw["hicp"], col, geo=geo, coicop=item), on="date", how="left")
        else:
            df = df.with_columns(pl.lit(None, dtype=pl.Float64).alias(col))

    rates = raw["rates"]
    m12 = "IRT_M12" if rates.filter(pl.col("int_rt") == "IRT_M12").height else "IRT_M3"
    df = df.join(tidy(rates, "euribor", int_rt=m12), on="date", how="left")

    # House prices are quarterly: compute growth per quarter, then give each month its quarter's value
    if "hpi" in raw and raw["hpi"].filter(pl.col("geo") == geo).height:
        hpi = tidy(raw["hpi"], "hpi", geo=geo).with_columns(yoy("hpi", lag=4).alias("hpi_yoy"))
        df = df.sort("date").join_asof(hpi, on="date", strategy="backward", tolerance="62d")
    else:
        df = df.with_columns(pl.lit(None, dtype=pl.Float64).alias("hpi"),
                             pl.lit(None, dtype=pl.Float64).alias("hpi_yoy"))

    base2019 = lambda c: pl.col(c) / pl.col(c).filter(pl.col("date").dt.year() == 2019).mean() * 100
    df = df.with_columns(
        yoy("hicp").alias("infl"),
        yoy("food").alias("food_infl"),
        yoy("rent").alias("rent_infl"),
        yoy("energy").alias("energy_infl"),
        (pl.col("unemp") - pl.col("unemp").shift(12)).alias("d_unemp"),
        *[base2019(c).alias(f"{c}_2019") for c in ["hicp", "food", "rent", "energy", "hpi"]],
    )
    return df.filter(pl.col("conf").is_not_null())

We save the result, so you (or a classmate without internet access) can reload it later. If the live download fails and a saved copy exists, the notebook uses the copy.

In [ ]:
CACHE = "spain_vibes_monthly.csv"

if all(k in raw for k in ["conf", "unemp", "hicp", "rates"]):
    es, ea = build("ES"), build(EA)
    pl.concat([es.with_columns(pl.lit("ES").alias("geo")),
               ea.with_columns(pl.lit("EA").alias("geo"))], how="diagonal_relaxed").write_csv(CACHE)
    print(f"Saved {CACHE}")
elif os.path.exists(CACHE):
    panel = pl.read_csv(CACHE, try_parse_dates=True, infer_schema_length=None)
    es = panel.filter(pl.col("geo") == "ES").drop("geo")
    ea = panel.filter(pl.col("geo") == "EA").drop("geo")
    print(f"Live download incomplete; loaded {CACHE}")
else:
    raise RuntimeError("Could not download the core series and no saved copy exists.")

print(f"Spain: {es['date'].min()} to {es['date'].max()}, {es.height} months")
es.tail(6)

## 3. Numbers vs vibes: a first look

Consumer confidence is a *balance*: the share of households answering positively minus the share answering negatively, averaged across four questions. Zero means as many optimists as pessimists; it is usually negative.

Put it next to the unemployment rate. For most of the history, the two move as mirror images. Look closely at what happens after 2020.

In [ ]:
fig, ax1 = plt.subplots()
ax1.plot(es["date"], es["conf"], color=ES_COLOR, label="Consumer confidence (left)")
ax1.set_ylabel("Confidence, balance")
ax2 = ax1.twinx()
ax2.plot(es["date"], es["unemp"], color=EA_COLOR, label="Unemployment rate (right, inverted)")
ax2.invert_yaxis()
ax2.set_ylabel("Unemployment, % (inverted)")
ax1.set_title("Spain: consumer confidence and unemployment")
fig.legend(loc="lower left", bbox_to_anchor=(0.08, 0.12), frameon=False)
plt.show()

In [ ]:
def period_means(df, cols):
    periods = [("2000-2007 boom", 2000, 2007), ("2008-2013 crisis", 2008, 2013),
               ("2014-2019 recovery", 2014, 2019), ("2020-2021 COVID", 2020, 2021),
               ("2022-2023 inflation", 2022, 2023), ("2024-now", 2024, 2100)]
    out = []
    for label, a, b in periods:
        sub = df.filter(pl.col("date").dt.year().is_between(a, b))
        out.append({"period": label, **{c: round(sub[c].mean(), 1) if sub[c].null_count() < sub.height else None
                                        for c in cols}})
    return pl.DataFrame(out)


period_means(es, ["conf", "unemp", "infl", "euribor"])

**Question.** In which period was unemployment highest? Confidence lowest? Compare the latest period with 2014 to 2019. If you only saw the unemployment column, what would you guess about confidence?

## 4. Inflation is a rate; prices are a level

Inflation has come down since its 2022 peak. But falling *inflation* means prices are rising more slowly, not that they are falling. Households remember what things used to cost. Here is the **price level** of different items relative to 2019.

In [ ]:
fig, ax = plt.subplots()
for col, label, c in [("hicp_2019", "All items", "black"), ("food_2019", "Food", "#e67e22"),
                      ("rent_2019", "Rents", "#8e44ad"), ("energy_2019", "Energy", "#16a085"),
                      ("hpi_2019", "House prices", ES_COLOR)]:
    sub = es.filter(pl.col("date") >= date(2015, 1, 1))
    if sub[col].null_count() < sub.height:
        ax.plot(sub["date"], sub[col], label=label, color=c, lw=2 if col == "hicp_2019" else 1.5)
ax.axhline(100, color="0.6", lw=0.8)
ax.set_ylabel("2019 = 100")
ax.set_title("Spain: price levels relative to 2019")
ax.legend(frameon=False)
plt.show()

latest = es.drop_nulls("hicp_2019").tail(1)
for col, label in [("hicp_2019", "All items"), ("food_2019", "Food"), ("rent_2019", "Rents"), ("hpi_2019", "House prices")]:
    v = es.drop_nulls(col).tail(1)
    if v.height:
        print(f"{label:13s} {v[col][0] - 100:+5.1f}% since 2019 (as of {v['date'][0]})")

**Caveat on rents.** The consumer price index measures rents for *all* tenants, most of whom are in existing contracts that rise slowly. People looking for a new flat face market rents that have risen far faster. Official rent inflation therefore understates the experience of young people and movers, which is exactly the group most likely to be unhappy.

## 5. Do people feel inflation accurately?

The survey also asks households how they think prices have changed over the past year. Compare that *perceived* inflation (a balance, not a percent) with the actual measured rate.

In [ ]:
fig, ax1 = plt.subplots()
ax1.plot(es["date"], es["infl"], color="black", label="Measured inflation, % (left)")
ax1.set_ylabel("Measured inflation, %")
ax2 = ax1.twinx()
ax2.plot(es["date"], es["prices_felt"], color=ES_COLOR, alpha=0.8, label="Perceived price rises, balance (right)")
ax2.set_ylabel("Perceived, balance")
ax1.set_title("Spain: measured vs perceived inflation")
fig.legend(loc="upper left", bbox_to_anchor=(0.08, 0.88), frameon=False)
plt.show()

both = es.drop_nulls(["infl", "prices_felt"])
for label, a, b in [("2000-2019", 2000, 2019), ("2020-now", 2020, 2100)]:
    sub = both.filter(pl.col("date").dt.year().is_between(a, b))
    print(f"Correlation {label}: {np.corrcoef(sub['infl'], sub['prices_felt'])[0, 1]:.2f}")

**Question.** When measured inflation fell back in 2023 and 2024, how quickly did *perceived* inflation follow? What might explain the lag?

## 6. A machine learning model of the vibes

Now the main event. We train models that predict monthly consumer confidence from four economic fundamentals:

* `unemp`: the unemployment rate
* `d_unemp`: its change over the past 12 months (people react to direction, not just level)
* `infl`: the inflation rate
* `euribor`: the 12-month Euribor, a proxy for mortgage payments

We train **only on 2000 to 2019**. Then we ask each model to predict 2020 onwards, data it has never seen. If actual confidence is far below the prediction, people feel worse than the fundamentals alone would suggest: a vibecession.

We compare three models:

* **Linear regression**: one coefficient per feature; easy to interpret.
* **Random forest** and **gradient boosting**: flexible tree-based models that can capture curves and interactions.

For a fair score on the training years we use **time-series cross-validation**: always train on the past and test on the following stretch, never the other way round.

In [ ]:
BASE_FEATURES = ["unemp", "d_unemp", "infl", "euribor"]
TRAIN_END = date(2019, 12, 1)

MODELS = {
    "Linear regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Random forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=0),
    "Gradient boosting": GradientBoostingRegressor(random_state=0),
}


def fit_vibes(df, features, train_start=date(2000, 1, 1), train_end=TRAIN_END):
    data = df.drop_nulls(features + ["conf"]).filter(pl.col("date") >= train_start)
    train = data.filter(pl.col("date") <= train_end)
    X, y = train.select(features).to_numpy(), train["conf"].to_numpy()
    X_all = data.select(features).to_numpy()

    preds, scores, fitted = {}, [], {}
    for name, proto in MODELS.items():
        model = clone(proto)
        cv_rmse = -cross_val_score(model, X, y, cv=TimeSeriesSplit(n_splits=5),
                                   scoring="neg_root_mean_squared_error").mean()
        model.fit(X, y)
        preds[name] = model.predict(X_all)
        post = data["date"] > train_end
        post_rmse = float(np.sqrt(np.mean((preds[name][post.to_numpy()] - data["conf"].to_numpy()[post.to_numpy()]) ** 2)))
        scores.append({"model": name, "CV error 2000-2019": round(cv_rmse, 2), "error after 2019": round(post_rmse, 2)})
        fitted[name] = model

    out = data.select("date", "conf").with_columns([pl.Series(n, p) for n, p in preds.items()])
    print(f"Trained on {train['date'].min()} to {train['date'].max()} ({train.height} months)")
    return out, pl.DataFrame(scores), fitted


es_fit, es_scores, es_models = fit_vibes(es, BASE_FEATURES)
es_scores

The errors are in the units of the confidence balance. Compare the error during training years with the error after 2019: a big jump means the post-2019 world behaves differently from what the model learned.

In [ ]:
def plot_fit(fit, title, start=date(2005, 1, 1)):
    sub = fit.filter(pl.col("date") >= start)
    fig, ax = plt.subplots()
    ax.axvspan(TRAIN_END, sub["date"].max(), color="0.94", zorder=0)
    ax.plot(sub["date"], sub["conf"], color="black", lw=2, label="Actual confidence")
    for name, ls in [("Linear regression", "--"), ("Random forest", ":")]:
        ax.plot(sub["date"], sub[name], ls=ls, color=MODEL_COLOR if ls == "--" else EA_COLOR, label=f"Predicted: {name}")
    ax.text(TRAIN_END, ax.get_ylim()[1], "  not used in training", va="top", fontsize=9, color="0.4")
    ax.set_ylabel("Confidence, balance")
    ax.set_title(title)
    ax.legend(frameon=False, loc="lower left")
    plt.show()


plot_fit(es_fit, "Spain: actual vs predicted consumer confidence")

### The vibes gap

Gap = actual minus predicted. Negative means people feel **worse** than the fundamentals suggest.

In [ ]:
def gap_table(fit):
    rows = []
    for label, a, b in [("2020-2021", 2020, 2021), ("2022-2023", 2022, 2023), ("2024-now", 2024, 2100)]:
        sub = fit.filter(pl.col("date").dt.year().is_between(a, b))
        rows.append({"period": label, **{n: round((sub["conf"] - sub[n]).mean(), 1) for n in MODELS}})
    return pl.DataFrame(rows)


gap_table(es_fit)

In [ ]:
lin = es_models["Linear regression"]
coefs = lin.named_steps["linearregression"].coef_
pl.DataFrame({"feature": BASE_FEATURES, "effect of a 1 s.d. increase": np.round(coefs, 2)})

**Reading the coefficients.** Because features are standardized, each number is the change in predicted confidence when that feature rises by one standard deviation (of its 2000 to 2019 values). Do the signs make sense?

### A warning about trees

Random forests and gradient boosting predict by averaging training examples that look similar. They **cannot extrapolate**: if 2022 inflation is higher than anything seen in 2000 to 2019, the tree models treat it as the highest value they *did* see. Check whether that applies here.

In [ ]:
train = es.filter(pl.col("date").is_between(date(2000, 1, 1), TRAIN_END))
post = es.filter(pl.col("date") > TRAIN_END)
pl.DataFrame({
    "feature": BASE_FEATURES,
    "train min": [round(train[f].min(), 1) for f in BASE_FEATURES],
    "train max": [round(train[f].max(), 1) for f in BASE_FEATURES],
    "post-2019 min": [round(post[f].min(), 1) for f in BASE_FEATURES],
    "post-2019 max": [round(post[f].max(), 1) for f in BASE_FEATURES],
})

## 7. Can housing and the shopping basket explain the gap?

Our first model only knew about the *overall* inflation rate. But people buy food every week and pay rent every month; they rarely buy a television. Let us add:

* `food_infl`: food price inflation
* `rent_infl`: rent inflation
* `hpi_yoy`: house price growth

Spain's house price series starts in 2007, so this model can only be trained from 2008 to 2019. That window still includes the housing bust and the recovery, so prices moved a lot.

In [ ]:
EXT_FEATURES = BASE_FEATURES + ["food_infl", "rent_infl", "hpi_yoy"]
ext_fit, ext_scores, ext_models = fit_vibes(es, EXT_FEATURES, train_start=date(2008, 1, 1))
print(ext_scores)

base_short, _, _ = fit_vibes(es, BASE_FEATURES, train_start=date(2008, 1, 1))  # same window, fair comparison
pl.concat([
    gap_table(base_short).select("period", pl.col("Linear regression").alias("basic model gap")),
    gap_table(ext_fit).select(pl.col("Linear regression").alias("with food & housing gap")),
], how="horizontal")

**Question.** Did adding food and housing make the gap smaller? If not, why might a model trained on 2008 to 2019 still fail to capture today's housing frustration? (Hint: in those years, falling house prices were bad news for homeowners; today rising prices are bad news for renters and would-be buyers. A model can only learn the relationship that existed in its training data.)

## 8. Is it just a European mood?

Maybe all of Europe feels gloomy, and Spain is no different. We run the same basic model for the euro area.

In [ ]:
comparison_features = [
    feature for feature in BASE_FEATURES
    if es[feature].is_not_null().any() and ea[feature].is_not_null().any()
]
if not comparison_features:
    raise ValueError("No BASE_FEATURES are available for both Spain and the euro area.")

print("Shared model features:", comparison_features)
es_compare_fit, _, _ = fit_vibes(es, comparison_features)
ea_fit, ea_scores, _ = fit_vibes(ea, comparison_features)
plot_fit(ea_fit, "Euro area: actual vs predicted consumer confidence")

compare = (
    es_compare_fit.select("date", (pl.col("conf") - pl.col("Linear regression")).alias("Spain"))
    .join(ea_fit.select("date", (pl.col("conf") - pl.col("Linear regression")).alias("Euro area")), on="date")
    .filter(pl.col("date") >= date(2015, 1, 1))
)
fig, ax = plt.subplots()
ax.plot(compare["date"], compare["Spain"], color=ES_COLOR, label="Spain")
ax.plot(compare["date"], compare["Euro area"], color=EA_COLOR, label="Euro area")
ax.axhline(0, color="black", lw=0.8)
ax.axvline(TRAIN_END, color="0.6", ls=":")
ax.set_ylabel("Actual minus predicted")
ax.set_title("The vibes gap: Spain vs the euro area (linear model)")
ax.legend(frameon=False)
plt.show()

## 9. What drives the mood? The survey components

The headline index averages four questions. Splitting them shows *which* part of the mood is off: people's view of their **own** finances, or of the **country's** economy? A classic vibecession pattern is "I'm doing OK, but the economy is terrible."

In [ ]:
comp = ["fin_past", "fin_next", "econ_next", "unemp_fear"]
labels = {"fin_past": "Own finances, past year", "fin_next": "Own finances, next year",
          "econ_next": "Country's economy, next year", "unemp_fear": "Expect unemployment to rise"}
fig, ax = plt.subplots()
sub = es.filter(pl.col("date") >= date(2010, 1, 1))
for c in comp:
    ax.plot(sub["date"], sub[c], label=labels[c])
ax.axhline(0, color="black", lw=0.6)
ax.set_ylabel("Balance")
ax.set_title("Spain: components of consumer confidence")
ax.legend(frameon=False, fontsize=9)
plt.show()

period_means(es, ["fin_past", "fin_next", "econ_next", "unemp_fear"])

## 10. Conclusions and discussion

Write a short paragraph answering the title question, using numbers from your own run.

1. **Verdict.** Based on the gap tables, is Spain in a vibecession? Since when? How big is the gap relative to the model's normal error?
2. **Model choice.** Which model would you trust most for predicting 2024 confidence, and why? Use both the cross-validation error and the extrapolation table.
3. **Missing variables.** Our models never see rents for *new* contracts, the share of young adults living with their parents, or social media. Which missing variable do you think matters most? Where could you find data for it?
4. **Back to the news.** A Spanish newspaper recently compared prime ministers and called housing Sánchez's "gran debe" (big shortfall). Does this notebook support that framing?
5. **Your turn.** Rerun the notebook with another country: change `"ES"` in the `build` call to `"PT"`, `"IT"` or `"DE"`. Is Spain unusual?